# 01 — Data Understanding and Preprocessing

This notebook prepares the raw football datasets for the World Cup 2026 prediction workflow.

## Project Setup

The project paths and helper functions are initialized.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_DIR / "src"))

from prepare_data import (
    read_table,
    normalize_columns,
    standardize_matches,
    standardize_fixtures,
    create_teams_from_fixtures,
    create_h2h_summary,
)

print("Project folder:", PROJECT_DIR)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)


Project folder: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026
Raw data folder: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\raw
Processed data folder: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed


## Raw Data Overview

The available source files are loaded and inspected.

In [2]:
raw_files = sorted([p for p in RAW_DIR.glob("*") if p.suffix.lower() in [".csv", ".xlsx", ".xls"]])

print("Files found in data/raw:")
for p in raw_files:
    print(" -", p.name)


Files found in data/raw:
 - all_matches.csv
 - countries_names.csv
 - FIFA2026_schedule.csv
 - FIFA2026_schedule_Fixtures.csv


## Column Inspection

The structure of each raw dataset is reviewed before preprocessing.

In [3]:
dataframes = {}

for file_path in raw_files:
    key = file_path.stem
    df = read_table(file_path)
    dataframes[key] = df

    print("=" * 100)
    print("FILE:", file_path.name)
    print("SHAPE:", df.shape)
    print("COLUMNS:")
    print(df.columns.tolist())
    display(df.head())


FILE: all_matches.csv
SHAPE: (51644, 8)
COLUMNS:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


FILE: countries_names.csv
SHAPE: (288, 4)
COLUMNS:
['original_name', 'current_name', 'color_code', 'secondary_color_code']


,original_name,current_name,color_code,secondary_color_code
0,Afghanistan,Afghanistan,#D32011,#FFFFFF
1,Albania,Albania,#FE0100,#000
2,Algeria,Algeria,#008000,#FFFFFF
3,Andorra,Andorra,#D50032,#FEDD00
4,Angola,Angola,#CC092F,#FFCB00


FILE: FIFA2026_schedule.csv
SHAPE: (104, 5)
COLUMNS:
['date', 'match_number', 'group', 'stadium', 'date_dt']


,date,match_number,group,stadium,date_dt
0,"Thursday, 11 June 2026",Match 1,Group A,Mexico City Stadium,2026-06-11
1,"Thursday, 11 June 2026",Match 2,Group A,Estadio Guadalajara,2026-06-11
2,"Friday, 12 June 2026",Match 3,Group B,Toronto Stadium,2026-06-12
3,"Friday, 12 June 2026",Match 4,Group D,Los Angeles Stadium,2026-06-12
4,"Saturday, 13 June 2026",Match 5,Group C,Boston Stadium,2026-06-13


FILE: FIFA2026_schedule_Fixtures.csv
SHAPE: (104, 6)
COLUMNS:
['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt']


,date,match_number,teams,group,stadium,date_dt
0,"Thursday, 11 June 2026",Match 1,Mexico v South Africa,Group A,Mexico City Stadium,2026-06-11
1,"Thursday, 11 June 2026",Match 2,Korea Republic v Czechia/Denmark/North Macedon...,Group A,Estadio Guadalajara,2026-06-11
2,"Friday, 12 June 2026",Match 3,Canada v Bosnia and Herzegovina/Italy/Northern...,Group B,Toronto Stadium,2026-06-12
3,"Friday, 12 June 2026",Match 4,USA v Paraguay,Group D,Los Angeles Stadium,2026-06-12
4,"Saturday, 13 June 2026",Match 5,Haiti v Scotland,Group C,Boston Stadium,2026-06-13


## Source Dataset Selection

The historical match dataset and the fixture dataset containing team pairings are selected.

In [4]:
if "all_matches" in dataframes:
    all_matches_raw = dataframes["all_matches"]
else:
    all_matches_key = max(dataframes.keys(), key=lambda k: len(dataframes[k]))
    all_matches_raw = dataframes[all_matches_key]
    print("Selected historical matches file:", all_matches_key)

fixture_candidates = []

for key, df in dataframes.items():
    cols = normalize_columns(df).columns.tolist()
    score = 0

    if "teams" in cols:
        score += 10
    if any(c in cols for c in ["team_a", "team1", "team_1", "home_team"]):
        score += 5
    if any(c in cols for c in ["team_b", "team2", "team_2", "away_team"]):
        score += 5
    if "match_number" in cols or "match_no" in cols:
        score += 2
    if "group" in cols:
        score += 2
    if "stadium" in cols:
        score += 1

    if "schedule" in key.lower() or "fixture" in key.lower():
        score += 1

    fixture_candidates.append((score, key, cols))

fixture_candidates = sorted(fixture_candidates, reverse=True)

print("Fixture candidate scores:")
for score, key, cols in fixture_candidates:
    print(score, key, cols)

fixtures_key = fixture_candidates[0][1]
fixtures_raw = dataframes[fixtures_key]

print("\nSelected fixtures file:", fixtures_key)
print("Historical matches shape:", all_matches_raw.shape)
print("Fixtures shape:", fixtures_raw.shape)


Fixture candidate scores:
16 FIFA2026_schedule_Fixtures ['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt']
10 all_matches ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'country', 'neutral']
6 FIFA2026_schedule ['date', 'match_number', 'group', 'stadium', 'date_dt']
0 countries_names ['original_name', 'current_name', 'color_code', 'secondary_color_code']

Selected fixtures file: FIFA2026_schedule_Fixtures
Historical matches shape: (51644, 8)
Fixtures shape: (104, 6)


## Historical Match Preprocessing

The historical match data is standardized, cleaned, and sorted chronologically.

In [5]:
clean_matches = standardize_matches(all_matches_raw)

print("Clean matches shape:", clean_matches.shape)
display(clean_matches.head())

clean_matches.to_csv(PROCESSED_DIR / "clean_all_matches.csv", index=False)
print("Saved:", PROCESSED_DIR / "clean_all_matches.csv")


Clean matches shape: (51644, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
0,1872-11-30,Scotland,England,0,0,Friendly,NaN,Scotland,False,Scotland,England,draw
1,1873-03-08,England,Scotland,4,2,Friendly,NaN,England,False,England,Scotland,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,NaN,Scotland,False,Scotland,England,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,NaN,England,False,England,Scotland,draw
4,1876-03-04,Scotland,England,3,0,Friendly,NaN,Scotland,False,Scotland,England,home_win


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\clean_all_matches.csv


## World Cup 2026 Fixture Preprocessing

The fixture data is standardized into a consistent match schedule format.

In [6]:
fixtures = standardize_fixtures(fixtures_raw)

print("Fixtures shape:", fixtures.shape)
display(fixtures.head())

fixtures.to_csv(PROCESSED_DIR / "wc2026_fixtures.csv", index=False)
print("Saved:", PROCESSED_DIR / "wc2026_fixtures.csv")


Fixtures shape: (104, 12)


,match_no,date,stage,group,team_a,team_b,team_a_clean,team_b_clean,venue,city,country,neutral
0,Match 1,2026-06-11,NaN,Group A,Mexico,South Africa,Mexico,South Africa,Mexico City Stadium,NaN,NaN,True
1,Match 2,2026-06-11,NaN,Group A,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,Estadio Guadalajara,NaN,NaN,True
2,Match 3,2026-06-12,NaN,Group B,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,Toronto Stadium,NaN,NaN,True
3,Match 4,2026-06-12,NaN,Group D,USA,Paraguay,USA,Paraguay,Los Angeles Stadium,NaN,NaN,True
4,Match 5,2026-06-13,NaN,Group C,Haiti,Scotland,Haiti,Scotland,Boston Stadium,NaN,NaN,True


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\wc2026_fixtures.csv


## Team List Extraction

The participating teams are extracted from the fixture table.

In [7]:
teams = create_teams_from_fixtures(fixtures)

print("Teams shape:", teams.shape)
display(teams.head(20))

teams.to_csv(PROCESSED_DIR / "wc2026_teams.csv", index=False)
print("Saved:", PROCESSED_DIR / "wc2026_teams.csv")


Teams shape: (112, 3)


,team,team_clean,group
0,Czechia/Denmark/North Macedonia/Republic of Ir...,Czechia/Denmark/North Macedonia/Republic of Ir...,Group A
1,Korea Republic,Korea Republic,Group A
2,Mexico,Mexico,Group A
3,South Africa,South Africa,Group A
4,Bosnia and Herzegovina/Italy/Northern Ireland/...,Bosnia and Herzegovina/Italy/Northern Ireland/...,Group B
5,Canada,Canada,Group B
6,Qatar,Qatar,Group B
7,Switzerland,Switzerland,Group B
8,Brazil,Brazil,Group C
9,Haiti,Haiti,Group C


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\wc2026_teams.csv


## World Cup Team Match History

Historical matches involving at least one World Cup 2026 team are filtered.

In [8]:
teams_list = sorted(teams["team_clean"].dropna().unique().tolist())

wc2026_related_matches = clean_matches[
    clean_matches["home_team_clean"].isin(teams_list) |
    clean_matches["away_team_clean"].isin(teams_list)
].copy()

print("Related matches shape:", wc2026_related_matches.shape)
display(wc2026_related_matches.head())

wc2026_related_matches.to_csv(PROCESSED_DIR / "wc2026_related_matches.csv", index=False)
print("Saved:", PROCESSED_DIR / "wc2026_related_matches.csv")


Related matches shape: (24541, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
0,1872-11-30,Scotland,England,0,0,Friendly,NaN,Scotland,False,Scotland,England,draw
1,1873-03-08,England,Scotland,4,2,Friendly,NaN,England,False,England,Scotland,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,NaN,Scotland,False,Scotland,England,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,NaN,England,False,England,Scotland,draw
4,1876-03-04,Scotland,England,3,0,Friendly,NaN,Scotland,False,Scotland,England,home_win


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\wc2026_related_matches.csv


## Head-to-Head Match History

Historical matches between two World Cup 2026 teams are filtered.

In [9]:
wc2026_head_to_head = clean_matches[
    clean_matches["home_team_clean"].isin(teams_list) &
    clean_matches["away_team_clean"].isin(teams_list)
].copy()

print("Head-to-head matches shape:", wc2026_head_to_head.shape)
display(wc2026_head_to_head.head())

wc2026_head_to_head.to_csv(PROCESSED_DIR / "wc2026_head_to_head.csv", index=False)
print("Saved:", PROCESSED_DIR / "wc2026_head_to_head.csv")


Head-to-head matches shape: (6329, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
0,1872-11-30,Scotland,England,0,0,Friendly,NaN,Scotland,False,Scotland,England,draw
1,1873-03-08,England,Scotland,4,2,Friendly,NaN,England,False,England,Scotland,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,NaN,Scotland,False,Scotland,England,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,NaN,England,False,England,Scotland,draw
4,1876-03-04,Scotland,England,3,0,Friendly,NaN,Scotland,False,Scotland,England,home_win


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\wc2026_head_to_head.csv


## Fixture-Level Head-to-Head Summary

Head-to-head statistics are calculated for each World Cup 2026 fixture.

In [10]:
h2h_summary = create_h2h_summary(fixtures, clean_matches)

print("Fixture H2H summary shape:", h2h_summary.shape)
display(h2h_summary.head(20))

h2h_summary.to_csv(PROCESSED_DIR / "wc2026_fixture_h2h_summary.csv", index=False)
print("Saved:", PROCESSED_DIR / "wc2026_fixture_h2h_summary.csv")


Fixture H2H summary shape: (104, 16)


,match_no,date,stage,group,team_a,team_b,team_a_clean,team_b_clean,h2h_matches,team_a_h2h_wins,draws,team_b_h2h_wins,team_a_h2h_goals,team_b_h2h_goals,team_a_avg_h2h_goals,team_b_avg_h2h_goals
0,Match 1,2026-06-11,NaN,Group A,Mexico,South Africa,Mexico,South Africa,4,2,1,1,10,5,2.500000,1.250000
1,Match 2,2026-06-11,NaN,Group A,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,0,0,0,0,0,0,0.000000,0.000000
2,Match 3,2026-06-12,NaN,Group B,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,0,0,0,0,0,0,0.000000,0.000000
3,Match 4,2026-06-12,NaN,Group D,USA,Paraguay,USA,Paraguay,9,5,2,2,12,7,1.333333,0.777778
4,Match 5,2026-06-13,NaN,Group C,Haiti,Scotland,Haiti,Scotland,0,0,0,0,0,0,0.000000,0.000000
5,Match 6,2026-06-13,NaN,Group D,Australia,Kosovo/Romania/Slovakia/Türkiye,Australia,Kosovo/Romania/Slovakia/Türkiye,0,0,0,0,0,0,0.000000,0.000000
6,Match 7,2026-06-13,NaN,Group C,Brazil,Morocco,Brazil,Morocco,3,2,0,1,6,2,2.000000,0.666667
7,Match 8,2026-06-13,NaN,Group B,Qatar,Switzerland,Qatar,Switzerland,1,1,0,0,1,0,1.000000,0.000000
8,Match 9,2026-06-14,NaN,Group E,Côte d'Ivoire,Ecuador,Côte d'Ivoire,Ecuador,0,0,0,0,0,0,0.000000,0.000000
9,Match 10,2026-06-14,NaN,Group E,Germany,Curaçao,Germany,Curaçao,0,0,0,0,0,0,0.000000,0.000000


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\wc2026_fixture_h2h_summary.csv


## Preprocessing Output

The processed datasets are saved for the next modelling steps.

In [11]:
print("Processed files:")
for p in sorted(PROCESSED_DIR.glob("*.csv")):
    print(" -", p.name)

print("Preprocessing completed.")


Processed files:
 - clean_all_matches.csv
 - wc2026_fixture_h2h_summary.csv
 - wc2026_fixtures.csv
 - wc2026_head_to_head.csv
 - wc2026_related_matches.csv
 - wc2026_teams.csv
Preprocessing completed.
